# Day 2 Smoke Test — Multi-Source Adapters & Extraction

Verifies that all Day 2 source adapters (**PDF**, **OCR fallback**, **CSV**, **SQL**) produce valid `RawDocument` contracts without swallowing errors or leaking normalization logic.

In [1]:
import sys, os
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(project_root))
os.chdir(project_root)
print("Working dir set to:", Path.cwd())

Working dir set to: d:\multi-source-rag-app


## 1. PDF Text Extraction (5-page clean text — no OCR)

In [2]:
from app.contracts import SourceDescriptor, SourceType
from app.sources.pdf import PDFSource

desc = SourceDescriptor(
    source_type=SourceType.PDF,
    name="orbital-text-manual",
    config={"path": "tests/fixtures/sample_text_5page.pdf"},
)
adapter = PDFSource(desc)
docs = [doc async for doc in adapter.fetch()]

print(f"Extracted {len(docs)} RawDocument(s)")
print("Total pages:", docs[0].metadata["total_pages"])
print("OCR triggered on any page:", any(p["ocr_used"] for p in docs[0].metadata["pages"]))
print("Snippet:", docs[0].raw_content[:120])

assert docs[0].metadata["total_pages"] == 5
assert not any(p["ocr_used"] for p in docs[0].metadata["pages"])
print("PASS — 5-page text PDF extracted cleanly without invoking OCR")

Extracted 1 RawDocument(s)
Total pages: 5
OCR triggered on any page: False
Snippet: [PAGE 1]
Orbital Robotics Specification Document - Page 1
This is official page 1 of the engineering manual.
PASS — 5-page text PDF extracted cleanly without invoking OCR


## 2. PDF Scanned Image Extraction (Per-page OCR fallback via Tesseract)

In [3]:
scanned_desc = SourceDescriptor(
    source_type=SourceType.PDF,
    name="scanned-memo",
    config={"path": "tests/fixtures/scanned_image.pdf"},
)
scanned_adapter = PDFSource(scanned_desc)
scanned_docs = [doc async for doc in scanned_adapter.fetch()]

page_info = scanned_docs[0].metadata["pages"][0]
print(f"Extracted {len(scanned_docs)} RawDocument(s)")
print("Page 1 OCR used:", page_info["ocr_used"])
print("Recovered OCR text:", repr(page_info["text"] if "text" in page_info else scanned_docs[0].raw_content.splitlines()[-1]))

assert page_info["ocr_used"] is True
assert "OCR fallback must extract this line" in scanned_docs[0].raw_content
print("PASS — Scanned image page correctly triggered OCR fallback")

Extracted 1 RawDocument(s)
Page 1 OCR used: True
Recovered OCR text: 'OCR fallback must extract this line'
PASS — Scanned image page correctly triggered OCR fallback


## 3. PDF Failure Path (Corrupt PDF raises typed `PDFExtractionError`)

In [4]:
from app.extraction.errors import PDFExtractionError
from app.extraction.pdf_extractor import extract_pdf

try:
    extract_pdf("tests/fixtures/corrupt.pdf")
    raise AssertionError("Should have failed on corrupt PDF")
except PDFExtractionError as exc:
    print("Caught expected error:", type(exc).__name__)
    print("Error code:", exc.error_code)
    print("Source reference:", exc.source_ref)
    assert exc.error_code in ("PDF_CORRUPT", "PDF_PARSE_ERROR")
    print("PASS — Corrupt PDF trapped with typed exception (ready for Day 4 DLQ)")

Caught expected error: PDFExtractionError
Error code: PDF_CORRUPT
Source reference: tests/fixtures/corrupt.pdf
PASS — Corrupt PDF trapped with typed exception (ready for Day 4 DLQ)


## 4. CSV Source (Schema-Agnostic: Reordered Columns & Unexpected Columns)

In [5]:
from app.sources.csv import CSVSource

# 1. Reordered columns
desc_reordered = SourceDescriptor(
    source_type=SourceType.CSV,
    name="reordered-csv",
    config={"path": "tests/fixtures/changed_order.csv"},
)
docs_reordered = [doc async for doc in CSVSource(desc_reordered).fetch()]
print(f"Reordered CSV: extracted {len(docs_reordered)} RawDocument rows")
print("Row 0 content:\n" + docs_reordered[0].raw_content)

# 2. Unexpected extra column
desc_extra = SourceDescriptor(
    source_type=SourceType.CSV,
    name="extra-col-csv",
    config={"path": "tests/fixtures/unexpected_column.csv"},
)
docs_extra = [doc async for doc in CSVSource(desc_extra).fetch()]
print(f"\nUnexpected column CSV: extracted {len(docs_extra)} RawDocument rows")
print("Row 0 extra column 'clearance_level' =", docs_extra[0].metadata["row_data"]["clearance_level"])

assert len(docs_reordered) == 2
assert "clearance_level" in docs_extra[0].metadata["row_data"]
print("PASS — CSV source extracts rows schema-agnostically")

Reordered CSV: extracted 2 RawDocument rows
Row 0 content:
department: Engineering
name: Alice
id: 101
salary: 95000

Unexpected column CSV: extracted 2 RawDocument rows
Row 0 extra column 'clearance_level' = TopSecret
PASS — CSV source extracts rows schema-agnostically


## 5. SQL Source (SQLAlchemy Table Extraction with Bounds)

In [6]:
from app.sources.sql import SQLSource
from pathlib import Path

db_path = Path("tests/fixtures/test_records.db").resolve()
sql_desc = SourceDescriptor(
    source_type=SourceType.SQL,
    name="employees-table",
    config={
        "connection_url": f"sqlite:///{db_path.as_posix()}",
        "table_name": "employees",
        "limit": 100,
    },
)
sql_adapter = SQLSource(sql_desc)
sql_docs = [doc async for doc in sql_adapter.fetch()]

print(f"Extracted {len(sql_docs)} SQL records from employees table:")
for d in sql_docs:
    row = d.metadata["row_data"]
    print(f"- [Row {d.metadata['row_index']}] {row['name']} | {row['role']} | {row['department']}")

assert len(sql_docs) == 3
assert sql_docs[0].metadata["table_name"] == "employees"
print("PASS — SQL table extracted with stable source and row metadata")

Extracted 3 SQL records from employees table:
- [Row 0] Elena Vance | Lead Propulsion Engineer | Propulsion
- [Row 1] Gordon Freeman | Research Scientist | Anomalous Materials
- [Row 2] Alyx Vance | Systems Integrator | Robotics
PASS — SQL table extracted with stable source and row metadata


## 6. Day 2 Gate Summary

In [7]:
print("=" * 59)
print("DAY 2 GATE: ALL PASS")
print("1. All 4 source paths (PDF text, PDF scan, CSV, SQL) -> valid RawDocument")
print("2. PDF OCR is a per-page fallback, not the default")
print("3. Corrupt PDF fails with typed error, no silent drop")
print("4. No adapter contains chunking/embedding/vector-store/LLM logic")
print("=" * 59)

DAY 2 GATE: ALL PASS
1. All 4 source paths (PDF text, PDF scan, CSV, SQL) -> valid RawDocument
2. PDF OCR is a per-page fallback, not the default
3. Corrupt PDF fails with typed error, no silent drop
4. No adapter contains chunking/embedding/vector-store/LLM logic
